# Active, nonreciprocal vector fitting at 190 GHz

This notebook applies Circulax's AAA fitting pipeline to the active transmitter
from scikit-rf's
[190 GHz example](https://scikit-rf.readthedocs.io/en/latest/examples/vectorfitting/vectorfitting_ex2_190ghz_active.html).
It is both an accuracy regression and a validation example: the data exposes
the assumptions that must be stated before a fitted model is admitted to
circuit simulation.

The measured device has gain and is directional. Therefore neither passivity
nor reciprocity is an acceptance requirement. Stability, accuracy, causality,
finite evaluation, and coverage of the intended simulation band remain
mandatory.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import skrf
from skrf.vectorFitting import VectorFitting

from circulax.fitting import (
    evaluate_sparameter_model,
    fit_with_delay,
    surface_from_fit,
    validate_surface_fit,
    y_to_s,
)

data_path = Path("examples/fitting/data/190ghz_tx_measured.s2p")
if not data_path.exists():
    data_path = Path("data/190ghz_tx_measured.s2p")
network = skrf.Network(data_path)
freqs = network.f
S = network.s
print(f"{len(freqs)} samples, {freqs[0] / 1e9:.0f}--{freqs[-1] / 1e9:.0f} GHz")

## Establish the intended physics

For a passive N-port, every singular value of its S-matrix is at most one.
That is not expected here: the device is an amplifier. Likewise, fitting only
one matrix triangle would erase the difference between forward and reverse
transmission.

In [ ]:
maximum_singular_value = np.linalg.svd(S, compute_uv=False)[..., 0].max()
maximum_nonreciprocity = np.max(np.abs(S[:, 0, 1] - S[:, 1, 0]))
best_reciprocal = 0.5 * (S + np.swapaxes(S, -1, -2))
reciprocal_error_floor = np.linalg.norm(best_reciprocal - S) / np.linalg.norm(S)

print(f"maximum singular value:       {maximum_singular_value:.3f}")
print(f"maximum |S21 - S12|:          {maximum_nonreciprocity:.3f}")
print(f"best reciprocal-model NRMSE:  {reciprocal_error_floor:.1%}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for row in range(2):
    for col in range(2):
        axes[row, col].plot(freqs / 1e9, 20 * np.log10(np.maximum(np.abs(S[:, row, col]), 1e-15)))
        axes[row, col].set_title(f"S{row + 1}{col + 1}")
        axes[row, col].set_ylabel("magnitude (dB)")
        axes[row, col].grid(True)
for axis in axes[-1]:
    axis.set_xlabel("frequency (GHz)")
fig.tight_layout()

## scikit-rf baselines

scikit-rf does **not** assume reciprocity: it fits every ordered response with
a common pole set. Passivity enforcement is a separate, optional operation.
The published example fits S directly. Circulax instead needs a rational Y
realization to stamp currents into its DAE, so the Y-domain baseline is the
fairer comparison for the current architecture.

In [ ]:
def skrf_response(parameter_type):
    fitter = VectorFitting(network)
    started = time.perf_counter()
    fitter.auto_fit(parameter_type=parameter_type)
    elapsed = time.perf_counter() - started
    response = np.empty_like(S)
    for row in range(2):
        for col in range(2):
            response[:, row, col] = fitter.get_model_response(row, col, freqs)
    if parameter_type == "y":
        response = np.stack([y_to_s(value, 50.0) for value in response])
    return fitter, response, elapsed


vf_s, S_skrf, time_skrf_s = skrf_response("s")
vf_y, S_skrf_y, time_skrf_y = skrf_response("y")

In [ ]:
def metrics(prediction):
    error = prediction - S
    return {
        "model NRMSE": np.linalg.norm(error) / np.linalg.norm(S),
        "RMS per S entry": np.sqrt(np.mean(np.abs(error) ** 2)),
        "maximum |dS|": np.max(np.abs(error)),
    }


for label, fitter, prediction, elapsed in (
    ("scikit-rf, direct S", vf_s, S_skrf, time_skrf_s),
    ("scikit-rf, Y then S", vf_y, S_skrf_y, time_skrf_y),
):
    result = metrics(prediction)
    order = fitter.get_model_order(fitter.poles)
    print(label)
    print(f"  order={order}, elapsed={elapsed:.3f} s")
    print("  " + ", ".join(f"{key}={value:.4g}" for key, value in result.items()))

## Circulax active-device fit

The required policies are explicit:

- `reciprocal=False` fits all four ordered responses rather than mirroring one triangle.
- `enforce_passive=False` preserves intentional gain.
- `delay_mode="auto"` resolves to `"none"` for a nonreciprocal device. A
  per-port reference-plane delay imposes the same delay in both transmission
  directions and is not justified by this measurement.

AAA discovers a stable initial topology from the dominant response. Common-pole
contribution ranking prunes weak conjugate pole pairs, and common-pole
vector-fitting iterations refine the reduced topology against every complex S
response. Finally, an exact state-space feedback transformation produces the Y
realization required by Circulax's circuit stamp without changing the S-domain
fit.

In [ ]:
started = time.perf_counter()
ss, delay, fit_metadata = fit_with_delay(
    S,
    freqs,
    tol=1e-8,
    mmax=40,
    reciprocal=False,
    enforce_passive=False,
    delay_mode="auto",
    fit_domain="s",
    s_refinement_iterations=4,
    max_poles=20,
    pole_count_candidates=tuple(range(10, 32, 2)),
    verbose=False,
)
time_circulax = time.perf_counter() - started
S_circulax = evaluate_sparameter_model(ss, freqs, delay)

print({key: value for key, value in fit_metadata.items() if key != "pole_sweep"})
print(f"elapsed={time_circulax:.3f} s")
print(metrics(S_circulax))

sweep = fit_metadata["pole_sweep"]
print()
print("Vmapped fixed-pole screening (before pole relocation):")
for count, error in zip(sweep.retained_counts, sweep.normalized_rmse, strict=True):
    print(f"  {int(count):2d} poles: {float(error):.3%} NRMSE")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex=True)
for row in range(2):
    for col in range(2):
        axis = axes[row, col]
        axis.plot(freqs / 1e9, 20 * np.log10(np.maximum(np.abs(S[:, row, col]), 1e-15)), label="measured")
        axis.plot(freqs / 1e9, 20 * np.log10(np.maximum(np.abs(S_circulax[:, row, col]), 1e-15)), "--", label="Circulax")
        axis.set_title(f"S{row + 1}{col + 1}")
        axis.set_ylabel("magnitude (dB)")
        axis.grid(True)
for axis in axes[-1]:
    axis.set_xlabel("frequency (GHz)")
axes[0, 0].legend()
fig.tight_layout()

## Apply the simulation gate

Wrapping the single-corner fit as a constant surface lets the same validation
report serve measured models and parameterized surfaces. Intentional activity
and nonreciprocity are declared, not inferred from a failed passive check.
The default accuracy thresholds remain unchanged. Accuracy now passes, while
the missing independent holdout sweep remains visible as a warning.

In [ ]:
omega_scale = 2 * np.pi * freqs[-1]
surface = surface_from_fit(ss, delay, omega_scale)
features = np.ones((1, 1))
report = validate_surface_fit(
    surface,
    S[None, ...],
    features,
    freqs,
    simulation_frequency_range=(freqs[0], freqs[-1]),
    expected_reciprocal=False,
    expected_passive=False,
)
print(report.summary())

## Release and simulation checklist

1. **Require independent validation.** Hold out frequency blocks or, preferably,
   a separate sweep. Report errors for every ordered S-parameter in addition
   to aggregate and worst-point errors.
2. **Restrict the simulation band.** The upstream example explicitly examines
   0--500 GHz because a good 140--220 GHz fit can behave implausibly when
   extrapolated. Validation must cover the entire requested simulation band.
3. **Retain the reduction regression.** Contribution-ranked pruning reduces
   the initial 50-pole topology to 20 poles and the exact transformed Y model
   from 100 to 40 stable states while retaining the 1.3% release error target.
4. **Keep active and passive policies separate.** Passive interconnect models
   still require dense-grid and asymptotic passivity checks. Active models do
   not, but they always require stable macromodel poles; system-level stability
   under the intended source and load terminations is a subsequent circuit
   check.

For passive data, refinement uses an S-domain loss with a Y-passivity penalty
and projects back onto sampled passivity after every optimizer update. If that
constraint pushes the error beyond its threshold, the correct response is to
increase or improve the pole set rather than silently relaxing passivity.

## Circulax suitability is different for an active device

This transmitter is intentionally active and nonreciprocal. A singular value above one is **not** a passive-model defect here, and passive enforcement would erase intended gain. Do not use this notebook to claim that scikit-rf fails because an amplifier is nonpassive.

Circulax still requires an appropriate, stable admittance realization and application-specific circuit stability checks. The cell below compares Y-realization stability for the scikit-rf S-domain and Y-domain fits; a failure means that particular realization is unsuitable as-is, not that every scikit-rf model is unsuitable. The validation report above remains authoritative for the Circulax fit, including its missing independent holdout evidence.

For new delay-free S fits, use `fit_model(S, freqs, options=ModelFitOptions(reciprocal=False, enforce_passivity=False))`, then `component_from_coefficients(coefficients)`. This is an API migration recipe, not a newly validated fit for this dataset. The existing advanced comparison is retained. See [API details](../../docs/fitting_api.md) and the [qualified passive ring-slot example](vector_fitting_00_ring_slot.ipynb).

Observed in the executed comparison: the scikit-rf S fit converts to Y with a largest pole real part of approximately **+2.71×10¹¹ rad/s** (unstable). Its direct Y fit has maximum pole real part **−4.76×10¹⁰ rad/s** (stable). The direct Y baseline therefore passes this stability check; it still requires application-specific accuracy and circuit validation. Intentional gain is not a failure.

In [ ]:
# The S fit needs conversion; the Y fit already represents admittance.
for label, fitted, needs_conversion in [
    ("scikit-rf S fit", vf_s, True),
    ("scikit-rf Y fit", vf_y, False),
]:
    A_ref, B_ref, C_ref, D_ref, E_ref = fitted._get_ABCDE()
    try:
        if needs_conversion:
            if np.any(E_ref != 0):
                raise ValueError("This diagnostic requires a proper S model.")
            A_ref = A_ref - B_ref @ np.linalg.solve(np.eye(len(D_ref)) + D_ref, C_ref)
        largest_real_pole = np.linalg.eigvals(A_ref).real.max()
        print(label, "Y max Re(pole):", largest_real_pole, "rad/s;",
              "stable" if largest_real_pole < 0 else "UNSTABLE")
    except (ValueError, np.linalg.LinAlgError) as error:
        print(label, "conversion check failed:", error)
print("Passivity is deliberately NOT an admission requirement for this active transmitter.")
